# 🔗 Penggabungan Data DBpedia & Wikidata

Notebook ini menggabungkan dua file CSV:
- `DBpedia_clean.csv` → kolom: `athleteName`, `clubName`
- `Wikidata_no_URI.csv` → kolom: `athleteLabel`, `clubLabel`

**Langkah-langkah:**
1. Upload file CSV
2. Load & preview masing-masing file
3. Unifikasi nama kolom
4. Gabungkan & hapus duplikat
5. Eksplorasi data hasil gabungan
6. Simpan ke CSV baru

## 📁 Step 1 — Upload File CSV

In [ ]:
from google.colab import files

print('Silakan upload DBpedia_clean.csv dan Wikidata_no_URI.csv')
uploaded = files.upload()
print('\nFile yang berhasil diupload:')
for fname in uploaded:
    print(f'  ✅ {fname}')

## 📊 Step 2 — Load & Preview Masing-masing CSV

In [ ]:
import pandas as pd

# Load masing-masing file
df_dbpedia = pd.read_csv('DBpedia_clean.csv')
df_wikidata = pd.read_csv('Wikidata_no_URI.csv')

print('=== DBpedia_clean.csv ===')
print(f'Jumlah baris : {len(df_dbpedia)}')
print(f'Kolom        : {df_dbpedia.columns.tolist()}')
print()
df_dbpedia.head()

In [ ]:
print('=== Wikidata_no_URI.csv ===')
print(f'Jumlah baris : {len(df_wikidata)}')
print(f'Kolom        : {df_wikidata.columns.tolist()}')
print()
df_wikidata.head()

## 🔄 Step 3 — Unifikasi Nama Kolom

Kedua CSV punya nama kolom berbeda, kita samakan menjadi `athlete` dan `club`.

In [ ]:
# Rename kolom DBpedia
df_dbpedia = df_dbpedia.rename(columns={
    'athleteName': 'athlete',
    'clubName':    'club'
})
df_dbpedia['source'] = 'DBpedia'

# Rename kolom Wikidata
df_wikidata = df_wikidata.rename(columns={
    'athleteLabel': 'athlete',
    'clubLabel':    'club'
})
df_wikidata['source'] = 'Wikidata'

print('DBpedia setelah rename:')
print(df_dbpedia.head(3))
print()
print('Wikidata setelah rename:')
print(df_wikidata.head(3))

## 🔗 Step 4 — Gabungkan & Hapus Duplikat

In [ ]:
# Gabungkan kedua dataframe
df_combined = pd.concat([df_dbpedia, df_wikidata], ignore_index=True)

print(f'Total baris sebelum dedup : {len(df_combined)}')

# Hapus nilai kosong
df_combined = df_combined.dropna(subset=['athlete', 'club'])

# Hapus duplikat berdasarkan pasangan athlete + club
duplikat = df_combined.duplicated(subset=['athlete', 'club']).sum()
print(f'Duplikat ditemukan        : {duplikat}')

df_combined = df_combined.drop_duplicates(subset=['athlete', 'club'])
df_combined = df_combined.reset_index(drop=True)

print(f'Total baris setelah dedup : {len(df_combined)}')
print()
df_combined.head(10)

## 🔍 Step 5 — Eksplorasi Data Hasil Gabungan

In [ ]:
print('=== Ringkasan Data ===')
print(f'Total relasi (atlet-klub) : {len(df_combined)}')
print(f'Atlet unik                : {df_combined["athlete"].nunique()}')
print(f'Klub unik                 : {df_combined["club"].nunique()}')
print()

print('=== Distribusi per Sumber ===')
print(df_combined['source'].value_counts())
print()

print('=== Jumlah Atlet per Klub ===')
print(df_combined.groupby('club')['athlete'].count().sort_values(ascending=False))

In [ ]:
import matplotlib.pyplot as plt

# Bar chart jumlah atlet per klub
club_counts = df_combined.groupby('club')['athlete'].count().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(club_counts.index, club_counts.values,
              color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])

# Tambahkan label nilai di atas bar
for bar, val in zip(bars, club_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontweight='bold')

ax.set_title('Jumlah Atlet per Klub (DBpedia + Wikidata)', fontsize=13, fontweight='bold')
ax.set_xlabel('Klub')
ax.set_ylabel('Jumlah Atlet')
ax.set_xticklabels(club_counts.index, rotation=15, ha='right')
plt.tight_layout()
plt.savefig('distribusi_atlet_per_klub.png', dpi=150)
plt.show()
print('Chart disimpan sebagai distribusi_atlet_per_klub.png')

In [ ]:
# Stacked bar: distribusi per sumber per klub
pivot = df_combined.groupby(['club', 'source']).size().unstack(fill_value=0)

pivot.plot(kind='bar', figsize=(8, 5), color=['#1f77b4', '#ff7f0e'],
           edgecolor='white')
plt.title('Distribusi Atlet per Klub berdasarkan Sumber Data', fontsize=12, fontweight='bold')
plt.xlabel('Klub')
plt.ylabel('Jumlah Atlet')
plt.xticks(rotation=15, ha='right')
plt.legend(title='Sumber')
plt.tight_layout()
plt.savefig('distribusi_per_sumber.png', dpi=150)
plt.show()
print('Chart disimpan sebagai distribusi_per_sumber.png')

## 💾 Step 6 — Simpan Hasil ke CSV

In [ ]:
output_file = 'combined_athlete_club.csv'
df_combined.to_csv(output_file, index=False)
print(f'✅ File berhasil disimpan: {output_file}')
print(f'   Total baris : {len(df_combined)}')
print(f'   Kolom       : {df_combined.columns.tolist()}')
print()
df_combined.head()

In [ ]:
# Download file hasil ke komputer lokal
files.download(output_file)
print('📥 File combined_athlete_club.csv sedang didownload...')